# DDPM do Zero no MNIST

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Modelos de difusão aprendem a *reverter* um processo de ruído. O processo forward corrompe os dados com ruído gaussiano; o reverso aprende a remover ruído passo a passo. Na inferência partimos de ruído puro e caminhamos de volta até uma amostra limpa.


## Formulação Matemática

Forward: $q(x_t \mid x_{t-1}) = \mathcal{N}(\sqrt{1-\beta_t}\,x_{t-1}, \beta_t I)$.

Forma fechada: $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, com $\bar\alpha_t = \prod_{s\leq t}(1-\beta_s)$.

Loss: $\mathbb{E}_{t, x_0, \epsilon}\,\big\|\epsilon - \epsilon_\theta(x_t, t)\big\|^2$.


## Implementação


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [ ]:
T = 200
beta = torch.linspace(1e-4, 0.02, T)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def q_sample(x0, t, noise):
    sqrt_ab = alpha_bar[t].sqrt().view(-1, 1, 1, 1)
    sqrt_1mab = (1 - alpha_bar[t]).sqrt().view(-1, 1, 1, 1)
    return sqrt_ab * x0 + sqrt_1mab * noise

class SmallUNet(nn.Module):
    """Tiny U-Net that conditions on the diffusion step."""
    def __init__(self):
        super().__init__()
        self.t_embed = nn.Sequential(nn.Linear(1, 32), nn.SiLU(), nn.Linear(32, 32))
        self.down = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.SiLU(),
                                  nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.SiLU())
        self.mid = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.SiLU())
        self.up = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.SiLU(),
                                nn.Conv2d(32, 1, 3, padding=1))
    def forward(self, x, t):
        te = self.t_embed(t.float().unsqueeze(-1) / T)
        h = self.down(x)
        h = h + te.view(-1, 32, 1, 1).repeat(1, 2, 1, 1)[:, :64]
        h = self.mid(h)
        return self.up(h)


## Experimento


In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
ds = datasets.MNIST('.', train=True, download=True, transform=tfm)
dl = DataLoader(ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SmallUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
alpha_bar = alpha_bar.to(device)

for epoch in range(1):  # bump for real training
    for x, _ in dl:
        x = x.to(device)
        t = torch.randint(0, T, (x.size(0),), device=device)
        noise = torch.randn_like(x)
        x_t = q_sample(x, t, noise)
        pred = model(x_t, t)
        loss = F.mse_loss(pred, noise)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'epoch {epoch} loss {loss.item():.4f}')


In [ ]:
@torch.no_grad()
def sample(n=8):
    x = torch.randn(n, 1, 28, 28, device=device)
    for t in reversed(range(T)):
        z = torch.randn_like(x) if t > 0 else 0
        eps = model(x, torch.full((n,), t, device=device))
        a_t = alpha[t].to(device); ab_t = alpha_bar[t].to(device); b_t = beta[t].to(device)
        x = (1/a_t.sqrt()) * (x - (b_t / (1 - ab_t).sqrt()) * eps) + b_t.sqrt() * z
    return x

samples = sample(8)
print('sample shape:', samples.shape)


## Discussão

- Mesmo com a U-Net minúscula daqui o loss deve cair rápido; com um modelo mais profundo e 30+ épocas começa a aparecer dígito legível.
- Schedules de variância: linear (acima), cosine (melhor para modelos pequenos), shifted-noise (alta resolução).
- DDIM é um sampler determinístico que troca qualidade por número de passos.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
